# Libraries Import

In [ ]:
import os, cv2, json, glob
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import tensorflow as tf
import tensorflow_hub as hub
from ultralytics import YOLO, RTDETR
# import pytorch as torch``

# from mmengine.config import Config
# from mmengine.registry import MODELS
# from mmengine.runner import load_checkpoint
# from mmaction.apis import init_recognizer

# # This function will now work correctly because we are running from the cloned directory
# from mmaction.utils import register_all_modules
# register_all_modules(init_default_scope=True) # We set the scope manually later

#Colab Base Path
# base_path = "/content/drive/MyDrive/SMT 6/CV/UAS"

#Local Base Path
base_path = ""

#Dataset Paths
# data_path = os.path.join(base_path, "match_videos")
# data_path = os.path.join(base_path, "practice_videos")
data_path = os.path.join(base_path, "ghost_battle")

# video_path = os.path.join(data_path, "Knee(Bryan) vs Double(Law) TWT 2024.mp4")
# video_path = os.path.join(data_path, "Knee(Bryan) vs Double(Law) 1 round.mp4")
video_path = os.path.join(data_path, "Lee_L_combo.mp4")

# annotation_path = os.path.join(base_path, "match_videos/Knee(Bryan) vs Double(Law) TWT 2024.json")
annotation_path = os.path.join(data_path, "Knee_reindexed.json")

output_dir = os.path.join(data_path, "frames")
kp_dir = os.path.join(data_path)

# video_path = os.path.join(base_path, "Bryan_2/Bryan_15_move_trimmed.mp4")
# annotation_path = os.path.join(base_path, "Bryan_2/Bryan_15_move_2.json")
# output_dir = os.path.join(base_path, "Bryan_2/frames")

# Load movenet
movenet = hub.load("https://tfhub.dev/google/movenet/singlepose/lightning/4").signatures['serving_default']
# yolo = YOLO("runs/detect/ghost_1/weights/best.pt")
# yolo.to("mps")

# detr  = RTDETR("runs/detect/twt_detr/weights/best.pt")
detr = RTDETR("rtdetr-l.pt")
detr.to("mps")

os.makedirs(output_dir, exist_ok=True)

RuntimeError: invalid low watermark ratio 1.4

# Create Fighter Dataset

In [4]:
def yolo_detect_batch(frame_batch, original_width, original_height):
    # Dynamic minimum size (5% of the image dimensions)
    min_w = original_width * 0.05
    min_h = original_height * 0.05

    # Run batch inference! (Process multiple frames at once)
    results = yolo(frame_batch, conf=0.3, iou=0.3, classes=[0], verbose=False)
    
    batch_filtered_boxes = []
    
    for result in results:
        if result.boxes is None or len(result.boxes) == 0:
            batch_filtered_boxes.append([])
            continue
            
        boxes = result.boxes.xyxy.cpu().numpy().astype(int)
        filtered_boxes = []
        
        for box in boxes:
            x1, y1, x2, y2 = box
            w = x2 - x1
            h = y2 - y1
            # Use the dynamic thresholds instead of hardcoded 200
            if w > min_w and h > min_h: 
                filtered_boxes.append(box)
                
        batch_filtered_boxes.append(np.array(filtered_boxes))
        
    return batch_filtered_boxes

def pad_box(h, w, box, padding=25):
    x1, y1, x2, y2 = box
    x1 = max(0, x1 - padding)
    y1 = max(0, y1 - padding)
    x2 = min(w, x2 + padding)
    y2 = min(h, y2 + padding)

    return np.array([x1, y1, x2, y2], dtype=int)

def xyxy_to_yolo_normalized(box, img_width, img_height):
    x1, y1, x2, y2 = box
    
    # Calculate width and height
    w = x2 - x1
    h = y2 - y1
    
    # Calculate center x and center y
    x_center = x1 + (w / 2)
    y_center = y1 + (h / 2)
    
    # Normalize by image dimensions
    x_center_n = x_center / img_width
    y_center_n = y_center / img_height
    w_n = w / img_width
    h_n = h / img_height
    
    return [x_center_n, y_center_n, w_n, h_n]

def process_video_batched(video_path, frame_dir, label_dir, frame_step=15, batch_size=16):
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    frame_batch = []
    frame_info_batch = [] # To keep track of frame numbers
    
    # Read the first frame to get dimensions
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    ret, test_frame = cap.read()
    if ret:
        original_height, original_width = test_frame.shape[:2]
    else:
        return

    for frame_count in range(0, total_frames, frame_step):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_count)
        ret, frame_bgr = cap.read()
        if not ret: break

        frame_batch.append(frame_bgr)
        frame_info_batch.append(frame_count)

        # When the batch is full, or we hit the end of the video
        if len(frame_batch) == batch_size or frame_count + frame_step >= total_frames:
            
            # Detect on the entire batch at once
            batch_boxes = yolo_detect_batch(frame_batch, original_width, original_height)
            
            for i, box_array in enumerate(batch_boxes):
                curr_frame_num = frame_info_batch[i]
                curr_frame_img = frame_batch[i]
                
                base_filename = f"{video_name}_frame_{curr_frame_num:05d}"
                image_path = os.path.join(frame_dir, f"{base_filename}.jpg")
                label_path = os.path.join(label_dir, f"{base_filename}.txt")

                # SAVE WITH COMPRESSION TO SAVE SSD SPACE
                cv2.imwrite(image_path, curr_frame_img, [cv2.IMWRITE_JPEG_QUALITY, 90])

                with open(label_path, "w") as f:
                    if len(box_array) > 0:
                        # padded_boxes = np.array([pad_box(original_height, original_width, box) for box in box_array], dtype=int)
                        padded_boxes = box_array

                        for box in padded_boxes:
                            norm_box = xyxy_to_yolo_normalized(box, original_width, original_height)
                            x_c, y_c, w, h = [max(0.0, min(1.0, val)) for val in norm_box] # Clamp
                            f.write(f"0 {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}\n")

            # Clear the lists for the next batch
            frame_batch = []
            frame_info_batch = []

    cap.release()

input_size = 192

frame_dir = os.path.join(base_path, "fighter_dataset/images")
label_dir = os.path.join(base_path, "fighter_dataset/labels")

os.makedirs(frame_dir, exist_ok=True)
os.makedirs(label_dir, exist_ok=True)

video_exts = ("*.mp4", "*.mkv")
videos = []
for ext in video_exts:
    videos.extend(glob.glob(os.path.join(data_path, ext)))
for v in sorted(videos):
    print("Processing", v)
    process_video_batched(v, frame_dir, label_dir, 15)

Processing ghost_battle/Knee(Bryan) vs Double(Law) TWT 2024.mp4


[h264 @ 0x3b54be390] mmco: unref short failure
[h264 @ 0x3b54be390] mmco: unref short failure
[h264 @ 0x3b54be390] mmco: unref short failure
[h264 @ 0x3b54be390] mmco: unref short failure
[h264 @ 0x3b54be390] mmco: unref short failure
[h264 @ 0x3b54be390] mmco: unref short failure
[h264 @ 0x3b54be390] mmco: unref short failure
[h264 @ 0x3b54be390] mmco: unref short failure
[h264 @ 0x3b54be390] mmco: unref short failure
[h264 @ 0x3b54be390] mmco: unref short failure
[h264 @ 0x3b54be390] mmco: unref short failure
[h264 @ 0x3b54be390] mmco: unref short failure
[h264 @ 0x3b54be390] mmco: unref short failure
[h264 @ 0x3b54be390] mmco: unref short failure
[h264 @ 0x3b54be390] mmco: unref short failure
[h264 @ 0x3b54be390] mmco: unref short failure
[h264 @ 0x3b54be390] mmco: unref short failure
[h264 @ 0x3b54be390] mmco: unref short failure
[h264 @ 0x3b54be390] mmco: unref short failure
[h264 @ 0x3b54be390] mmco: unref short failure
[h264 @ 0x3b54be390] mmco: unref short failure
[h264 @ 0x3b5

Processing ghost_battle/bryan_dragunov.mp4
Processing ghost_battle/claudio_lars.mp4
Processing ghost_battle/jin_hwoarang.mp4
Processing ghost_battle/kazuya_lee.mp4
Processing ghost_battle/law_hwoarang.mp4
Processing ghost_battle/paul_alisa.mp4
Processing ghost_battle/shaheen_steve.mp4
Processing ghost_battle/steve_leroy.mp4
Processing ghost_battle/victor_azu.mp4


# Train Val Split

In [4]:
import os
import shutil
import random
import yaml

# Adjust base_path if needed
base_path = ""
dataset_dir = os.path.join(base_path, "fighter_dataset")  # Change to your dataset directory

# Source directories from your previous step
src_images_dir = os.path.join(dataset_dir, "images")
src_labels_dir = os.path.join(dataset_dir, "labels") # Note: YOLO prefers 'labels' plural, we will fix this below

# YOLO standard split directories
train_images_dir = os.path.join(dataset_dir, "images", "train")
val_images_dir = os.path.join(dataset_dir, "images", "val")
train_labels_dir = os.path.join(dataset_dir, "labels", "train")
val_labels_dir = os.path.join(dataset_dir, "labels", "val")

# Ensure target directories exist
for dir_path in [train_images_dir, val_images_dir, train_labels_dir, val_labels_dir]:
    os.makedirs(dir_path, exist_ok=True)

# Get all image files and shuffle them for a random split
image_files = [f for f in os.listdir(src_images_dir) if f.endswith('.jpg')]
random.shuffle(image_files)

# Define split ratio (80% Train, 20% Val)
split_ratio = 0.8
split_index = int(len(image_files) * split_ratio)

train_files = image_files[:split_index]
val_files = image_files[split_index:]

def populate_split(file_list, dest_img_dir, dest_lbl_dir):
    for img_file in file_list:
        # Source paths
        src_img = os.path.join(src_images_dir, img_file)
        
        lbl_file = img_file.replace('.jpg', '.txt')
        src_lbl = os.path.join(src_labels_dir, lbl_file)
        
        # Destination paths
        dest_img = os.path.join(dest_img_dir, img_file)
        dest_lbl = os.path.join(dest_lbl_dir, lbl_file)
        
        # Move files
        if os.path.exists(src_img):
            shutil.move(src_img, dest_img)
        if os.path.exists(src_lbl):
            shutil.move(src_lbl, dest_lbl)

# Execute the split
populate_split(train_files, train_images_dir, train_labels_dir)
populate_split(val_files, val_images_dir, val_labels_dir)

# Clean up the old, now empty 'label' directory if you want
if os.path.exists(src_labels_dir) and not os.listdir(src_labels_dir):
    os.rmdir(src_labels_dir)

# Generate the data.yaml file required by YOLO
yaml_content = {
    'path': os.path.abspath(dataset_dir),
    'train': 'images/train',
    'val': 'images/val',
    'nc': 1,
    'names': {0: 'fighter'}
}

yaml_path = os.path.join(dataset_dir, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_content, f, sort_keys=False)

print(f"✅ Split Complete! {len(train_files)} Training | {len(val_files)} Validation")
print(f"✅ data.yaml generated at: {yaml_path}")

✅ Split Complete! 844 Training | 211 Validation
✅ data.yaml generated at: fighter_dataset/data.yaml


# Yolo Training

In [ ]:
yolo.train(
    data='fighter_dataset/data.yaml',
    epochs=100,
    imgsz=640,
    batch=32,           # Increased to utilize 24GB RAM and stabilize gradients
    device='mps',
    patience=20,        # Give it a bit more room to breathe
    workers=8,          # M-series chips handle 8 workers beautifully
    half=False,         # Disabled to prevent MPS gradient stalling
    cache=True,
    cos_lr=True,        # Use Cosine Learning Rate to gently unstick plateaus 
    
    # --- TAILORED AUGMENTATIONS ---
    # --- DISABLED (Breaks Fighting Game Logic) ---
    mosaic=0.0,      # Characters shouldn't be sliced into 4 quadrants
    mixup=0.0,       # Ghosts/transparency ruin pose estimation prep
    degrees=0.0,     # Camera pitch doesn't roll
    perspective=0.0, # Camera stays relatively fixed in 3D fighters
    flipud=0.0,      # No upside-down fighting
    
    # --- ENABLED (Forces Generalization) ---
    fliplr=0.5,      # MUST KEEP: Simulates Player 1 side vs Player 2 side
    translate=0.1,   # KEEP: Simulates camera panning left/right
    scale=0.3,       # KEEP: Handles dynamic zoom-ins during combos/rage arts
    hsv_h=0.015,     # KEEP: Alters hue to help generalize across the 9 different maps
    hsv_s=0.7,       # KEEP: Color saturation variation
    hsv_v=0.4        # KEEP: Brightness variation
)

# RT-DETR Training

In [4]:
detr.train(
    data='fighter_dataset_ghost_splited_1k/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,           # Increased to utilize 24GB RAM and stabilize gradients
    device='mps',
    patience=20,        # Give it a bit more room to breathe
    workers=4,          # M-series chips handle 8 workers beautifully
    half=False,         # Disabled to prevent MPS gradient stalling
    cache=True,
    cos_lr=True,        # Use Cosine Learning Rate to gently unstick plateaus 
)

New https://pypi.org/project/ultralytics/8.4.46 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.38 🚀 Python-3.10.11 torch-2.11.0 MPS (Apple M5)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=fighter_dataset_ghost_splited_1k/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train6, 

RuntimeError: invalid low watermark ratio 1.4